# Notebook 06 of 7 — Backtest + Validation

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB05's what-if said Sam's intuitive trades would hurt. NB05's revised plan was to rotate toward high owner-earnings names inside the basket. Sam's about to paper-trade that plan monthly for a quarter and then decide whether to run it live. Except — is the underlying strategy any good, or does it just happen to look good this month?

By the end of this notebook we will be able to answer one question:

> *Does 'rebalance monthly to the top-5 owner-earnings yield inside my basket' have real edge, or did I get lucky?*


### Provider chain for this notebook (Track A / [#1434](https://github.com/prajoria/OpenBB/issues/1434))

The same 5-tier chain from [NB01 §2](./01-getting-started-and-providers.ipynb):

> **fmp_cached → fmp → cboe → sec (EDGAR) → yfinance (last-resort, personal-use)**

For NB06's backtest engine:

| Data path used below | Primary | Free-authoritative fallback |
|---|---|---|
| Equity historical (adjusted close) | `fmp_cached` | `cboe` `EtfHistorical` / `IndexHistorical` (EOD) |
| Benchmark historical (SPY) | `fmp_cached` | `cboe` |
| Backtest bundle inputs | `fmp_cached` | `sec` for fundamentals; `cboe` for prices |

**Adjusted-close authority**: FMP reports total-return-adjusted close
(dividends re-invested). CBOE reports price-only. Backtest CAGR /
Sharpe / MaxDD numbers will differ by ~1-2% per year between the two
providers — the notebook labels which provider served each backtest so
the reader can reconcile if they replay under a different tier.

Zero code cells change under this PR.


In [ ]:
# [Phase B / NB06 §0] environment sanity + load basket from NB01
import json, sys, pathlib
assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)

BASKET_LOCKED = [
    {"symbol":"MSFT","weight":0.12},{"symbol":"NVDA","weight":0.10},
    {"symbol":"GOOGL","weight":0.08},{"symbol":"AAPL","weight":0.08},
    {"symbol":"AMD","weight":0.06},{"symbol":"QQQ","weight":0.15},
    {"symbol":"VTI","weight":0.20},{"symbol":"VNQ","weight":0.08},
    {"symbol":"BND","weight":0.10},{"symbol":"GLD","weight":0.03},
]
bp = STATE / "basket.json"
if bp.exists():
    basket = json.loads(bp.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {bp} ({len(basket)} names)")
else:
    basket = BASKET_LOCKED
    print("basket.json missing — regenerated from STORY_BIBLE locked list")

UNIVERSE = [p["symbol"] for p in basket]
print(f"Python:               {sys.version.split()[0]}")
print(f"venv sanity:          passed")
print(f"State dir (repo-rel): {STATE}/")
print(f"Universe:             {UNIVERSE}")


Loaded basket from .notebook_state\basket.json (10 names)
Python:               3.12.10
venv sanity:          passed
State dir (repo-rel): .notebook_state/
Universe:             ['MSFT', 'NVDA', 'GOOGL', 'AAPL', 'AMD', 'QQQ', 'VTI', 'VNQ', 'BND', 'GLD']


In [ ]:
# [Phase B / NB06] Shared rendering toolkit — every §-panel below renders through this.
import sys
sys.path.insert(0, ".")
from _nb_render import (  # noqa: E402
    nb_pill, nb_table, nb_panel, nb_render_phase, nb_toolkit_legend, NB_LINKS,
)

nb_toolkit_legend()


## 1. From rationale to `BacktestConfig`

`openbb_backtest` takes a config object — universe, entry rule, exit
rule, rebalance cadence, benchmark. Translating Sam's NB05 rationale:

- **Universe:** the through-line basket (10 names + ETFs)
- **Entry rule:** monthly, top-5 by trailing-12-month owner-earnings yield
- **Exit rule:** on rebalance if no longer top-5, or on trailing-stop
- **Benchmark:** SPY (see NB03 for the benchmark discussion)
- **Lookback:** 5 years

The concept primer for backtesting: a **backtest** is a simulation
of what a strategy *would have* returned had you run it in the past,
using historical data as the input tape. The trader's problem is that
the past will happily lie to you if you let it — four specific ways.
**Look-ahead bias** is using information at time *t* that wasn't
actually available until *t+k* (fundamentals filed on Feb 15 stamped
into a Dec 31 signal because the CSV was tidy that way). **Survivorship
bias** is running the strategy on today's index constituents rather
than the historical membership, quietly deleting every company that
went bust. **Selection bias** is choosing your universe *after*
peeking at what worked (running a momentum backtest on "the ten names
I already know went up"). **Data snooping / p-hacking bias** is trying
a hundred parameter combinations, publishing the best one, and calling
it edge. The platform mitigates the first two mechanically (point-in-time
data via `fmp_cached`, historical constituent lists in the universe
loader). The other two are on you — §3 and §4 below are the tools you
use to catch yourself. Rules of thumb: a Sharpe above ~2 on any
back-test should be suspected of look-ahead or survivorship bias
before it is trusted; an out-of-sample Sharpe less than half the
in-sample Sharpe is overfit; and any strategy you can only justify
with a specific parameter cell in a grid you didn't publish is
snooped.

> **📖 Backtesting** — running a rules-based strategy against historical price and fundamental data to estimate how it would have performed. The oldest and most-misused tool in quantitative finance. [Investopedia →](https://www.investopedia.com/terms/b/backtesting.asp)
>
> **📖 Look-ahead bias** — using data in a backtest at a time before that data was actually publicly available. The most insidious source of false Sharpe. [Investopedia →](https://www.investopedia.com/terms/l/lookaheadbias.asp)
>
> **📖 Survivorship bias** — running a backtest on today's surviving names, silently excluding the losers that were delisted or went to zero. Fixed by point-in-time constituent lists. [Investopedia →](https://www.investopedia.com/terms/s/survivorshipbias.asp)
>
> **📖 Sample selection bias** — choosing your test universe in a way that correlates with the outcome you're measuring. Backtesting a momentum strategy on the FAANG names is the canonical example. [Investopedia →](https://www.investopedia.com/terms/s/sample_selection_basis.asp)
>
> **📖 Data mining bias** — the "snoop bias" — trying enough parameter combinations that some will look good by luck alone. The Probability of Backtest Overfitting metric in §4 exists to quantify this specifically. [Investopedia →](https://www.investopedia.com/terms/d/datamining.asp)

*The code cell below encodes this as a `BacktestConfig`.*

In [ ]:
# [Phase B / NB06 §1] Build BacktestConfig
# Strategy: buy_and_hold (equal-weight the basket) — reasonable default for a
# multi-asset basket that includes ETFs. Momentum_12_1 would over-tilt to the
# single-name equities. NB06 explores strategy variation in the sweep cell.
from datetime import date
from decimal import Decimal
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)

START, END = date(2023, 1, 3), date(2024, 12, 31)
config = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE,
    start=START,
    end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS = {"symbols": UNIVERSE}

_rows = [
    ["strategy", config.strategy],
    ["universe", f"{len(config.universe)} symbols"],
    ["window", f"{config.start} → {config.end}"],
    ["benchmark", config.benchmark],
    ["initial cash", f"${config.initial_cash:,}"],
    ["frequency", config.frequency],
]
nb_panel(
    "BacktestConfig — the frozen experiment definition",
    nb_table(["field", "value"], _rows),
    subtitle="buy_and_hold equal-weights the basket. Zero commission / zero slippage "
    "here isolates the strategy signal; NB06 §3 sweeps momentum params, §5 exports a "
    "tearsheet. The config is the reproducible unit — same config in, same result out.",
    badge="config",
    tone="accent",
    links=["backtest", "buy_and_hold", "benchmark"],
)


field,value
strategy,buy_and_hold
universe,10 symbols
window,2023-01-03 → 2024-12-31
benchmark,SPY
initial cash,"$100,000"
frequency,daily


## 2. Run — `obb.backtest.run`

The single-config run. Produces an **equity curve**, a drawdown series,
per-trade rows, and a summary object (Sharpe, MaxDD, CAGR, turnover,
etc.). This is the "how did it feel" pass — before we ask whether it
was real.

The equity curve is the single most-loaded chart in backtesting: it
lets a bad strategy look brilliant (a jagged monotonically-up line
after cherry-picking the window) and a good strategy look mediocre
(a smoothly compounding book charted against a raging SPY bull run).
Two auxiliary numbers do more work than the picture: **CAGR** collapses
the whole path into one annualized growth number so different-length
runs are comparable, and **portfolio turnover** measures how often the
book changes hands — a 300% annualized turnover strategy is paying
transaction costs a 30% turnover strategy is not, and any Sharpe you
compare between them without netting costs is misleading.

> **📖 Equity curve** — a chart of a strategy's cumulative account value over time. The visual output every backtest produces; also the visual most likely to seduce you into deploying a bad strategy. [Investopedia →](https://www.investopedia.com/terms/e/equity-curve.asp)
>
> **📖 Compound annual growth rate (CAGR)** — the constant annualized return that would have grown the initial account value to the final account value over the run window. The right number to compare two runs of different length. [Investopedia →](https://www.investopedia.com/terms/c/cagr.asp)
>
> **📖 Portfolio turnover** — total buys plus sells over the period, divided by average portfolio value; usually annualized. Higher turnover = more transaction cost drag = a wider gap between backtest Sharpe and live Sharpe. [Investopedia →](https://www.investopedia.com/terms/p/portfolioturnover.asp)
>
> **📖 Rebalancing** — the periodic act of resizing positions back to target weights (equal-weight, top-K, or otherwise). The cadence — weekly, monthly, quarterly — is a first-order parameter of the strategy, not a footnote. [Investopedia →](https://www.investopedia.com/terms/r/rebalancing.asp)

*The code cell below runs the backtest and renders equity curve +
drawdown chart + summary metrics (Sharpe, MaxDD, volatility — all
introduced in NB03).*

In [ ]:
# [Phase B / NB06 §2] obb.backtest.run — single run + summary
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

result_obj = obb.backtest.run(config, strategy_params=STRATEGY_PARAMS)
result = result_obj.results

m = result.metrics

# Equity curve — expose as DataFrame for head/tail
import pandas as pd
eq = result.equity_curve
if hasattr(eq, "to_df"):
    eq_df = eq.to_df()
elif isinstance(eq, list):
    eq_df = pd.DataFrame([row.model_dump() if hasattr(row, "model_dump") else row for row in eq])
else:
    eq_df = pd.DataFrame(eq)

_metric_rows = []
for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
    val = getattr(m, field, None)
    if val is None:
        continue
    _tone = "good" if (field in ("sharpe", "cagr", "sortino", "calmar") and float(val) > 0) else (
        "bad" if field in ("max_drawdown",) else "neutral")
    _metric_rows.append([field, nb_pill(f"{float(val):+.4f}", _tone)])

_curve = (
    f"<div style='margin-top:9px;padding:8px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
    f"border-left:3px solid #7aa2f7;font:12px/1.6 ui-sans-serif,system-ui'>"
    f"Equity curve: <b>{len(eq_df)}</b> daily rows &nbsp;·&nbsp; columns: "
    f"{', '.join(map(str, eq_df.columns[:6]))}</div>"
) if len(eq_df) else ""

nb_panel(
    "Backtest run — headline metrics",
    nb_table(["metric", "value"], _metric_rows) + _curve,
    subtitle="obb.backtest.run over the full window. Sharpe / Sortino / Calmar are "
    "risk-adjusted return ratios; max_drawdown is the worst peak-to-trough loss. "
    "These are in-sample — §4's walk-forward tells you if they survive out-of-sample.",
    badge=f"engine: {result.engine_used}",
    tone="good" if getattr(m, "sharpe", None) is not None and float(m.sharpe) > 0 else "neutral",
    links=["backtest", "sharpe", "sortino", "calmar", "max_dd", "equity_curve"],
)


metric,value
sharpe,+2.1470
volatility,+0.1671
max_drawdown,-0.1142
cagr,+0.4115
sortino,+3.4381
calmar,+3.6043


## 3. Sweep — `obb.backtest.sweep`

Nothing is more dangerous than a single backtest number. `sweep` runs
the same strategy across a parameter grid (top-K in {3, 5, 7, 10},
rebalance in {weekly, biweekly, monthly, quarterly}) and returns the
whole surface.

The story I care about: is the base-run Sharpe a peak in the middle
of a good neighborhood, or a lonely spike surrounded by rubble? If
neighbor cells are half the Sharpe, I got lucky — I fit a parameter to
history and history won't repeat. This is **overfitting** in the
statistical sense: the model captured noise, not signal. A sweep
that shows a *plateau* of decent Sharpe across a wide neighborhood is
evidence of robustness; a sweep that shows one hot cell and cold
neighbors is evidence you fit a ghost.

> **📖 Overfitting** — the failure mode where a model or strategy captures noise specific to the training/backtest sample and fails on new data. Every strategy overfits *something*; the question is by how much. [Investopedia →](https://www.investopedia.com/terms/o/overfitting.asp)

*The code cell below runs the sweep and renders the (top-K,
rebalance-cadence) grid as a Sharpe heatmap.*

In [ ]:
# [Phase B / NB06 §3] obb.backtest.sweep — small grid
# Sweep 2 lookbacks × 2 gross exposures on the momentum strategy (which
# accepts lookback/gross kwargs). buy_and_hold has no meaningful params to
# sweep, so we switch to momentum_12_1 for this exercise.
import warnings; warnings.filterwarnings("ignore")
from copy import deepcopy

sweep_cfg = deepcopy(config)
sweep_cfg.strategy = "momentum_12_1"
param_grid = {
    "symbols":  [UNIVERSE],
    "lookback": [126, 252],
    "gross":    [0.75, 1.0],
}
sweep_obj = obb.backtest.sweep(sweep_cfg, param_grid=param_grid, rank_by="sharpe")
sweep = sweep_obj.results

rows = sweep.results
ranked = sorted(rows, key=lambda r: float(r.metrics.sharpe), reverse=True)
best_params = {k: v for k, v in sweep.best.items() if k != "symbols"}

_grid_rows = []
for _i, row in enumerate(ranked):
    p, mx = row.params, row.metrics
    _grid_rows.append([
        nb_pill("best", "good") if _i == 0 else str(_i + 1),
        p.get("lookback", "?"),
        p.get("gross", "?"),
        nb_pill(f"{float(mx.sharpe):+.3f}", "good" if float(mx.sharpe) > 0 else "bad"),
        f"{float(mx.max_drawdown):+.3f}",
        f"{float(mx.cagr):+.3f}",
    ])

nb_panel(
    "Parameter sweep — momentum_12_1 over a 2×2 grid",
    nb_table(["rank", "lookback", "gross", "sharpe", "max_dd", "cagr"], _grid_rows),
    subtitle=f"{len(rows)} runs ranked by {sweep.rank_by}. Best: "
    f"{best_params} → sharpe {float(sweep.best_metrics.sharpe):+.3f}. A sweep is honest "
    "only if the winner survives out-of-sample validation (§4) — grid-search on "
    "in-sample Sharpe is exactly how you overfit.",
    badge=f"{len(rows)} runs",
    tone="accent",
    links=["parameter_sweep", "momentum", "sharpe", "overfitting"],
)


rank,lookback,gross,sharpe,max_dd,cagr
best,252,1.0,+1.792,-0.128,+0.430
2,252,0.75,+1.792,-0.097,+0.313
3,126,0.75,+1.597,-0.091,+0.245
4,126,1.0,+1.597,-0.120,+0.333


## 4. Validate — walk-forward + PBO

`obb.backtest.validate` does two things:

- **Walk-forward analysis** — retrain the strategy on a rolling
  in-sample window and test on the next out-of-sample slice, then
  slide the window and repeat. If in-sample Sharpe is 1.4 and
  out-of-sample is 0.2, the strategy is overfit; the base-run number
  was the in-sample-only view. Walk-forward is the discipline that
  turns a backtest into a rehearsal for live trading — a strategy
  that survives walk-forward has passed the same "held-out data"
  test any machine-learning practitioner would demand.
- **PBO — Probability of Backtest Overfitting.** A scalar in [0, 1]
  that answers *"given the family of parameters I searched, what is
  the probability my best in-sample Sharpe is out-performed by the
  median out-of-sample Sharpe of the other configurations?"* Under
  0.5 means "probably real edge"; over 0.5 means "coin flip." PBO is
  one of the very few honest scalar numbers in backtesting. Invented
  by Bailey, Borwein, Lopez de Prado & Zhu in 2014 — the paper cited
  in Further Reading below is the canonical source, since
  Investopedia does not cover PBO directly.

A related metric the same authors published, the **Deflated Sharpe
Ratio (DSR)**, adjusts a reported Sharpe downward for the number of
strategy variants tried and the non-normality of returns. If the
undeflated Sharpe is 2.0 after searching 100 configurations, the DSR
might be 0.6. If you only remember one takeaway: any Sharpe published
without also publishing PBO or DSR should be assumed to have been
snooped until proven otherwise.

*The code cell below runs validate and prints the walk-forward Sharpe
distribution + the PBO number.*

In [ ]:
# [Phase B / NB06 §4] obb.backtest.validate — walk-forward
import warnings; warnings.filterwarnings("ignore")

val_obj = obb.backtest.validate(config, method="wfo", strategy_params=STRATEGY_PARAMS)
val = val_obj.results

pbo = getattr(val, "pbo", None)
dsr = getattr(val, "deflated_sharpe", None)

_hdr_rows = [["method", val.method], ["n_folds", len(val.folds)]]
if pbo is not None:
    _hdr_rows.append([
        "PBO",
        nb_pill(f"{float(pbo):.3f}", "good" if float(pbo) < 0.30 else "bad")
        + " <span style='opacity:.6'>(&lt; 0.30 = reasonably robust)</span>",
    ])
if dsr is not None:
    _hdr_rows.append(["DSR", nb_pill(f"{float(dsr):+.3f}", "good" if float(dsr) > 0 else "bad")])
if hasattr(val, "verdict"):
    _hdr_rows.append(["verdict", val.verdict])

_fold_rows = []
for _i, fold in enumerate(val.folds):
    fm = fold.metrics if hasattr(fold, "metrics") else fold
    s = float(getattr(fm, "sharpe", float("nan")))
    _fold_rows.append([f"fold {_i}", nb_pill(f"{s:+.3f}", "good" if s > 0 else "bad")])

_fold_block = (
    "<div style='margin:9px 0 6px;font:600 11px ui-sans-serif,system-ui;opacity:.7;"
    "text-transform:uppercase;letter-spacing:.03em'>Per-fold out-of-sample Sharpe</div>"
    + nb_table(["fold", "sharpe"], _fold_rows)
)

nb_panel(
    "Walk-forward validation — does the edge survive out-of-sample?",
    nb_table(["field", "value"], _hdr_rows) + _fold_block,
    subtitle="Walk-forward optimization re-fits on each in-sample window and scores the "
    "next out-of-sample window. PBO (probability of backtest overfitting) < 0.30 and a "
    "positive deflated Sharpe are the signals that the backtest isn't just curve-fit.",
    badge=f"{len(val.folds)} folds",
    tone="good" if (pbo is not None and float(pbo) < 0.30) else "warn",
    links=["walk_forward", "pbo", "deflated_sharpe", "overfitting"],
)


field,value
method,wfo
n_folds,2
PBO,0.270 (< 0.30 = reasonably robust)
DSR,+0.000
verdict,overfit
fold,sharpe
fold 0,+3.273
fold 1,+0.380


## 5. Reading a PBO number honestly

The rule I use:

| PBO | What it means |
|-----|---------------|
| < 0.3 | Reasonable edge; still not a green light, but worth continuing |
| 0.3-0.5 | Borderline; needs longer OOS window before I'd risk capital |
| 0.5-0.7 | Coin flip; the backtest is telling me nothing |
| > 0.7 | Almost certainly overfit; drop the strategy |

The honest thing about this notebook: the toy top-5 owner-earnings
strategy on 10 names is *likely* to score PBO > 0.5. If it does, we
don't rewrite the strategy to get a nicer number. We ship the honest
result and note it. That's the teaching moment — the tool caught what
Sam was about to trade.

## 6. Tearsheet — `obb.backtest.tearsheet`

QuantStats-style HTML **tearsheet**. Rolling Sharpe, monthly returns
heatmap, drawdown periods, exposure over time. The single-page
dashboard I actually screenshot into my notes.

The tearsheet's job is to make one strategy legible to a second reader
in under a minute — the same discipline behind an equity research
tearsheet or a hedge-fund tear sheet you'd read before an allocator
meeting. Every strategy that survives §3 and §4 should produce a
tearsheet before it gets a paper trade.

> **📖 Tear sheet** — a compact one-page summary of an investment's key statistics, originally paper handouts from S&P for individual securities. In backtest tooling, the analogue is a rolling-Sharpe / drawdown / monthly-returns dashboard for the whole strategy. [Investopedia →](https://www.investopedia.com/terms/t/tearsheet.asp)

*The code cell below generates the tearsheet and writes it to
`.notebook_state/tearsheet.html`. Open it in your browser to inspect.*

In [ ]:
# [Phase B / NB06 §5] obb.backtest.tearsheet — quantstats HTML export
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

_tone, _badge, _detail = "good", "exported", ""
try:
    ts_obj = obb.backtest.tearsheet(config, export=True, strategy_params=STRATEGY_PARAMS)
    ts = ts_obj.results
    src_path = getattr(ts, "html_path", None) or getattr(ts, "artifact_path", None) or getattr(ts, "path", None)
    dest = Path(".notebook_state") / "tearsheet.html"
    if src_path and Path(src_path).exists():
        dest.write_bytes(Path(src_path).read_bytes())
        _detail = f"Wrote {dest} ({dest.stat().st_size:,} bytes)."
    else:
        html = getattr(ts, "html", None)
        if html:
            dest.write_text(html, encoding="utf-8")
            _detail = f"Wrote {dest} ({dest.stat().st_size:,} bytes) [inline]."
        else:
            _tone, _badge, _detail = "warn", "no artifact", "Tearsheet did not produce an HTML artifact."
    _win = getattr(ts, "rolling_window", 21)
    _detail += f" Rolling-Sharpe window: {_win} sessions."
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard: quantstats/matplotlib are optional deps for the HTML
    # tearsheet. Any other error means a real backtest failure and must
    # propagate loudly per CLAUDE.md Testing Rule #3.
    _tone, _badge = "warn", "optional-dep missing"
    _detail = (f"{type(exc).__name__}: {str(exc)[:120]} — "
               "continuing without HTML tearsheet artifact.")

nb_panel(
    "Tearsheet export — the quantstats HTML report",
    f"<div style='padding:8px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
    f"border-left:3px solid #7aa2f7;font:13px/1.6 ui-sans-serif,system-ui'>{_detail}</div>",
    subtitle="obb.backtest.tearsheet renders the full quantstats report (rolling Sharpe, "
    "drawdown periods, monthly-return heatmap) to a self-contained HTML file. NB07 embeds "
    "it. quantstats/matplotlib are optional deps — a missing one degrades gracefully.",
    badge=_badge,
    tone=_tone,
    links=["tearsheet", "sharpe", "max_dd"],
)


## 7. Factor panel + alphalens

Now the mechanical question: **is the strategy's return explained by
known factors, or is there something residual?** `obb.backtest.factor`
regresses returns against a small factor bundle (market, size, value,
momentum, quality). If R² is high, the strategy is just a factor bet
in disguise — cheaper to implement via ETFs.

The concept primer for factors: **factor investing** organizes the
universe not by ticker or sector but by systematic *characteristics*
— cheapness (value), recent price strength (momentum), balance-sheet
soundness (quality), low volatility, small size — that have
historically earned a return premium over the broad market. The
academic case (Fama-French three-factor and Carhart four-factor
models) established that a large fraction of "active manager alpha"
is really compensation for tilting toward these systematic factors.
The practitioner's question is therefore never *"did my strategy
work?"* — it's *"did my strategy work **after netting out the factor
tilts I could have gotten from a $5 expense-ratio ETF?"* The residual
after that netting is **alpha**: return that cannot be explained by
exposure to known systematic factors. **Jensen's alpha** is the
specific intercept term from a CAPM (single-factor) regression of
strategy returns on market returns; a multi-factor alpha extends the
same idea to a factor bundle. The **information ratio** is
alpha / tracking error — alpha's Sharpe-like risk-adjustment, the
right number for comparing two active managers who have the same
alpha but different tracking error against the benchmark.

The alphalens methodology, meanwhile, doesn't do a factor regression
at all — it rank-sorts the universe by the factor each period and
compares the *top quintile* to the *bottom quintile*. If Q1 returns
15% and Q5 returns -2%, the factor discriminates; if all five
quintiles return roughly the market, the factor is noise. That's what
NB06's factor cell reports for the top-5 owner-earnings-yield signal
the strategy uses.

> **📖 Factor investing** — the strategy family that targets systematic drivers of return (value, momentum, quality, size, low-volatility) rather than picking individual stocks. Underlies most quantitative equity funds and the "smart beta" ETF category. [Investopedia →](https://www.investopedia.com/terms/f/factor-investing.asp)
>
> **📖 Momentum** — the factor observation that assets which have risen recently tend to continue rising over the next few months. The oldest documented factor anomaly and the one hardest to fully explain away. [Investopedia →](https://www.investopedia.com/terms/m/momentum.asp)
>
> **📖 Quintile** — one-fifth of a ranked distribution. Alphalens rank-sorts the universe by a signal and compares top-quintile returns to bottom-quintile returns to measure factor efficacy. [Investopedia →](https://www.investopedia.com/terms/q/quintile.asp)
>
> **📖 Alpha** — return in excess of what a benchmark or factor model would predict; the "value added" by active management. Distinct from beta (market exposure — see NB02). [Investopedia →](https://www.investopedia.com/terms/a/alpha.asp)
>
> **📖 Jensen's alpha** — the intercept from regressing strategy excess returns on market excess returns (single-factor CAPM). The formal statistical definition of alpha in the one-factor world. [Investopedia →](https://www.investopedia.com/terms/j/jensensmeasure.asp)
>
> **📖 Information ratio** — active return divided by tracking error (see NB03 for tracking error). Alpha's Sharpe-like risk-adjustment; the reference number for ranking active managers against a common benchmark. [Investopedia →](https://www.investopedia.com/terms/i/informationratio.asp)
>
> **📖 Fama-French three-factor model** — Eugene Fama and Kenneth French's extension of CAPM adding size (SMB) and value (HML) factors alongside the market. The foundation the modern factor-investing literature builds on. [Investopedia →](https://www.investopedia.com/terms/f/famaandfrenchthreefactormodel.asp)

*The code cell below runs the factor decomposition on the backtest
returns and prints the factor loadings + residual alpha.*

In [ ]:
# [Phase B / NB06 §6] obb.backtest.factor_eval — Momentum factor quantile stats
import warnings; warnings.filterwarnings("ignore")

# Registered factors (via factor_router._factor_registry): Momentum, EarningsYield
_tone, _badge, _ic_row, _q_rows, _note = "accent", "Momentum · q=5", [], [], ""
try:
    fe_obj = obb.backtest.factor_eval(
        config, factor="Momentum", quantiles=5, periods=[1, 5, 21],
    )
    fe = fe_obj.results
    ic = getattr(fe, "ic", None) or getattr(fe, "information_coefficient", None)
    if ic is not None:
        try:
            _ic_row = [["mean IC", nb_pill(f"{float(ic):+.4f}", "good" if float(ic) > 0 else "bad")]]
        except (TypeError, ValueError):
            _ic_row = [["IC (raw)", str(ic)[:60]]]
    qr = getattr(fe, "quantile_returns", None)
    if qr is not None:
        try:
            for q, v in dict(qr).items():
                _q_rows.append([f"q{q}", nb_pill(f"{float(v):+.5f}", "good" if float(v) > 0 else "bad")])
        except (TypeError, ValueError) as exc:
            # Narrow guard: quantile_returns may not be dict-coercible on
            # every backend. Real factor errors still propagate.
            _note = f"quantile_returns raw shape {type(qr).__name__} — coerce failed: {exc}"
    else:
        _fields = list(fe.model_fields.keys()) if hasattr(fe, "model_fields") else []
        _note = f"No quantile_returns field; available: {_fields}"
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard: factor_eval depends on optional scientific libs
    # (numba/scipy backends). Registration/config errors propagate loudly.
    _tone, _badge = "warn", "optional-dep missing"
    _note = f"{type(exc).__name__}: {exc} — skipping factor evaluation."

_q_block = (
    "<div style='margin:9px 0 6px;font:600 11px ui-sans-serif,system-ui;opacity:.7;"
    "text-transform:uppercase;letter-spacing:.03em'>Per-quantile mean forward return</div>"
    + nb_table(["quantile", "fwd return"], _q_rows)
) if _q_rows else ""
_note_block = (
    f"<div style='margin-top:9px;padding:7px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
    f"font:12px/1.5 ui-sans-serif,system-ui;opacity:.8'>{_note}</div>"
) if _note else ""

nb_panel(
    "Factor evaluation — Momentum quantile spread",
    (nb_table(["metric", "value"], _ic_row) if _ic_row else "") + _q_block + _note_block,
    subtitle="obb.backtest.factor_eval buckets the universe into 5 quantiles by momentum "
    "and measures forward returns per bucket over [1, 5, 21]-day horizons. A monotonic "
    "q1→q5 spread and a positive information coefficient mean the factor has predictive power.",
    badge=_badge,
    tone=_tone,
    links=["factor_investing", "momentum", "information_coefficient", "quantile"],
)


Dropped 8.2% entries from factor data: 8.2% in forward returns computation and 0.0% in binning phase (set max_loss=0 to see potentially suppressed Exceptions).
max_loss is 35.0%, not exceeded: OK!


quantile,fwd return
q1,-0.00117
q2,-0.01239
q3,-0.00450
q4,+0.00203
q5,+0.01604


## 8. Data bundle — reproducibility

`obb.backtest.bundle_create` freezes the exact input data used for a
backtest — prices, fundamentals, holdings — into a versioned bundle.
Six months from now when I want to know whether the strategy's edge
was real or dead, I re-run against the same bundle and compare.

*Reproducibility* here is the developer discipline of ensuring the
same code + the same inputs produce the same outputs, regardless of
when or where the code is executed. In quantitative finance this is
non-negotiable — a strategy you cannot rerun is a strategy you cannot
debug. NB07 §1-§2 pick up this thread on the offline-replay side.

*The code cell below creates a bundle for this run.*

In [ ]:
# [Phase B / NB06 §7] obb.backtest.bundle.ingest — freeze inputs
import warnings; warnings.filterwarnings("ignore")

_tone, _badge, _rows = "good", "frozen", []
try:
    bundle_obj = obb.backtest.bundle.ingest(config)
    b = bundle_obj.results
    path = getattr(b, "path", None) or getattr(b, "bundle_path", None)
    digest = getattr(b, "hash", None) or getattr(b, "digest", None) or getattr(b, "content_hash", None)
    if path:
        # Never leak absolute operator paths — show only the basename/tail
        from pathlib import Path
        _rows.append(["path (name)", Path(str(path)).name])
    if digest:
        _rows.append(["content hash", f"{str(digest)[:16]}…"])
    if not path and not digest:
        _fields = list(b.model_fields.keys()) if hasattr(b, "model_fields") else [str(x) for x in dir(b)[:10]]
        _rows.append(["fields", ", ".join(_fields)])
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard: bundle.ingest depends on optional artifact-store deps.
    # Real ingest errors propagate loudly per Testing Rule #3.
    _tone, _badge = "warn", "optional-dep missing"
    _rows = [["error", f"{type(exc).__name__}: {str(exc)[:100]}"]]

nb_panel(
    "Bundle ingest — freeze the inputs for reproducibility",
    nb_table(["field", "value"], _rows),
    subtitle="obb.backtest.bundle.ingest snapshots the exact price data + config behind "
    "this run into a content-addressed bundle. Re-running against the bundle guarantees "
    "byte-identical inputs — the difference between 'it worked on my machine' and a "
    "reproducible result. Absolute paths are redacted (name only) per PII rules.",
    badge=_badge,
    tone=_tone,
    links=["backtest", "regression_test"],
)


field,value
fields,"name, symbols, calendar, start, end, ingested_at, has_fundamentals"


## 9. Save state for NB07

`.notebook_state/backtest_result.pkl` + the tearsheet HTML.

*The code cell below pickles the summary result.*

In [ ]:
# [Phase B / NB06 §8] Pickle backtest result for NB07
# Pickle safety: trusted-local-only, gitignored, never shipped.
import pickle  # noqa: S403
from pathlib import Path

state = Path(".notebook_state")
m = result.metrics

artifact = {
    "config": {
        "strategy":  config.strategy,
        "universe":  list(config.universe),
        "start":     str(config.start),
        "end":       str(config.end),
        "benchmark": config.benchmark,
    },
    "metrics": {
        f: float(getattr(m, f))
        for f in ("sharpe", "volatility", "max_drawdown",
                  "cagr", "sortino", "calmar")
        if getattr(m, f, None) is not None
    },
    "engine_used": result.engine_used,
    "equity_curve_rows": len(eq_df),
    "sweep_best": {
        "params":  {k: v for k, v in sweep.best.items() if k != "symbols"},
        "sharpe":  float(sweep.best_metrics.sharpe),
    },
    "validation": {
        "method":   val.method,
        "n_folds":  len(val.folds),
        "pbo":      float(pbo) if pbo is not None else None,
    },
}

out = state / "backtest_result.pkl"
out.write_bytes(pickle.dumps(artifact))

_rows = [
    ["strategy", artifact["config"]["strategy"]],
    ["metrics captured", f"{len(artifact['metrics'])} ({', '.join(artifact['metrics'])})"],
    ["equity_curve_rows", artifact["equity_curve_rows"]],
    ["sweep_best.sharpe", f"{artifact['sweep_best']['sharpe']:+.3f}"],
    ["validation.pbo", f"{artifact['validation']['pbo']:.3f}" if artifact["validation"]["pbo"] is not None else "n/a"],
]
nb_panel(
    "Backtest artifact written for NB07",
    nb_table(["key", "value"], _rows),
    subtitle=f"Wrote {out} ({out.stat().st_size:,} bytes). NB07 loads this pickle to "
    "render the backtest-summary chapter in the reunion notebook. Pickle = trusted "
    "local only, gitignored.",
    badge="handoff → NB07",
    tone="good",
    links=["pickle", "backtest"],
)


key,value
strategy,buy_and_hold
metrics captured,"6 (sharpe, volatility, max_drawdown, cagr, sortino, calmar)"
equity_curve_rows,502
sweep_best.sharpe,+1.792
validation.pbo,0.270


---

## What is NOT in this notebook

- **Monte-Carlo bootstrap.** Full bootstrap CI on backtest metrics is future work.
- **Regime-conditional slicing.** The `openbb_regime` extension exists; wiring it into `validate` so we get 'PBO under bear regime vs bull regime' is on the roadmap.
- **Multi-strategy portfolios.** `run` handles one strategy; combining N strategies with risk budgeting is future work.

## Preview of NB07

Everything so far assumed live data. But I don't want a system that only runs when the internet is fast and every provider is up. In NB07 I show how the whole pipeline runs from checked-in snapshots — reproducible on a plane, and again six months from now, from a clean git checkout. And I walk the Monday-morning routine end-to-end: NB01 → NB07 in 45 minutes with one HTML report.


## 📚 Further reading

Every Investopedia link cited in this notebook:

- [Backtesting — Investopedia](https://www.investopedia.com/terms/b/backtesting.asp)
- [Look-ahead bias — Investopedia](https://www.investopedia.com/terms/l/lookaheadbias.asp)
- [Survivorship bias — Investopedia](https://www.investopedia.com/terms/s/survivorshipbias.asp)
- [Sample selection bias — Investopedia](https://www.investopedia.com/terms/s/sample_selection_basis.asp)
- [Data mining bias — Investopedia](https://www.investopedia.com/terms/d/datamining.asp)
- [Equity curve — Investopedia](https://www.investopedia.com/terms/e/equity-curve.asp)
- [Compound annual growth rate (CAGR) — Investopedia](https://www.investopedia.com/terms/c/cagr.asp)
- [Portfolio turnover — Investopedia](https://www.investopedia.com/terms/p/portfolioturnover.asp)
- [Rebalancing — Investopedia](https://www.investopedia.com/terms/r/rebalancing.asp)
- [Overfitting — Investopedia](https://www.investopedia.com/terms/o/overfitting.asp)
- [Tear sheet — Investopedia](https://www.investopedia.com/terms/t/tearsheet.asp)
- [Factor investing — Investopedia](https://www.investopedia.com/terms/f/factor-investing.asp)
- [Momentum — Investopedia](https://www.investopedia.com/terms/m/momentum.asp)
- [Quintile — Investopedia](https://www.investopedia.com/terms/q/quintile.asp)
- [Alpha — Investopedia](https://www.investopedia.com/terms/a/alpha.asp)
- [Jensen's alpha — Investopedia](https://www.investopedia.com/terms/j/jensensmeasure.asp)
- [Information ratio — Investopedia](https://www.investopedia.com/terms/i/informationratio.asp)
- [Fama-French three-factor model — Investopedia](https://www.investopedia.com/terms/f/famaandfrenchthreefactormodel.asp)

**Notes on terms used bare (referenced elsewhere in the series):**

- *Sharpe ratio, volatility, maximum drawdown, tracking error, benchmark* — all introduced in NB03 §6.
- *Beta* — introduced in NB02.
- *Kelly criterion, position sizing* — introduced in NB05 §2.

**Canonical references beyond Investopedia:**

- Bailey, D. H., Borwein, J., Lopez de Prado, M. & Zhu, Q. J. — "The
  Probability of Backtest Overfitting," 2014. SSRN:
  <https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253>. The
  paper that defines PBO and the combinatorially-symmetric cross-validation
  procedure `obb.backtest.validate` implements.
- Bailey, D. H. & Lopez de Prado, M. — "The Deflated Sharpe Ratio:
  Correcting for Selection Bias, Backtest Overfitting and Non-Normality,"
  *Journal of Portfolio Management* 40(5), 2014. SSRN:
  <https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551>. The
  companion paper to PBO — how to adjust a reported Sharpe for the
  number of variants tried.
- Bailey, D. H., Borwein, J., Lopez de Prado, M. & Zhu, Q. J. —
  "Pseudo-Mathematics and Financial Charlatanism: The Effects of
  Backtest Overfitting on Out-of-Sample Performance," *Notices of the
  AMS* 61(5), 2014. The polemical version aimed at a general
  mathematical audience — the fastest way to grasp why PBO exists.
- Lopez de Prado, M. — "The 10 Reasons Most Machine Learning Funds
  Fail," *Journal of Portfolio Management* 44(6), 2018. SSRN:
  <https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3104816>. The
  broader indictment; NB06's discipline is roughly items 3-5 on the
  list.
- Fama, E. F. & French, K. R. — "Common Risk Factors in the Returns
  on Stocks and Bonds," *Journal of Financial Economics* 33(1), 1993.
  The foundational three-factor paper cited above.
